# Second Ear: keyword baseline

Scores the same held-out test split as the trained model with a fixed list of red-flag phrases.
It reports the same tests and metrics: precision, recall and F1 with n, on full dialogues and on their first 40 % of words, with the leading greeting removed (headline) and with the original text.
It does not train anything, tune the list on the test split, or print a dialogue.

**Run it:** Runtime > Run all. No GPU, no installs, no Drive. It takes a few seconds.

**Output:** `baselines.json`, same shape as `model.json`. Copy it to `<repo>/eval/results/baselines.json`.
Run from the repo root, it writes there directly.

Only the keyword baseline is here. The Haiku zero-shot baseline is not built in this version.

In [ ]:
# 1. Config. The dataset revision and the 40 % prefix match train_risk_model.ipynb.
DATASET_REPO = "BothBosu/scam-dialogue"
DATASET_REVISION = "321b961b5ae353e19ed479b960658dcd223d5e06"  # the revision checked on 2026-11-14
TEST_FILE = "scam-dialogue_test.csv"
EXPECTED_COLUMNS = ["dialogue", "type", "label"]
EXPECTED_TEST_ROWS = 320
EARLY_PERCENT = 40

# The fixed list from ml/README.md. It was set before the test split was scored and is not tuned.
KEYWORDS = (
    "gift card",
    "wire transfer",
    "bitcoin",
    "don't tell anyone",
    "remote access",
    "warrant",
    "medicare number",
)

In [ ]:
# Shared helpers. This cell is identical in train_risk_model.ipynb and baselines.ipynb.
import csv
import json
import os
import re
from collections import Counter
from datetime import datetime, timezone

from huggingface_hub import hf_hub_download

TAG_PATTERN = re.compile(r"\b(?:caller|receiver)\s*:\s*", re.IGNORECASE)
GREETING_PATTERN = re.compile(
    r"^\s*(?:hello|hi|hey|good\s+(?:morning|afternoon|evening))\b[\s,.!?;:\-\u2013\u2014]*", re.IGNORECASE
)
RESULTS_SCHEMA = "second-ear/results-v1"
METRIC_NAMES = ("precision", "recall", "f1")


def load_split(repo: str, revision: str, filename: str, expected_columns: list[str]) -> list[dict]:
    """Download one CSV at a pinned revision and return its rows, labels as ints."""
    path = hf_hub_download(repo, filename, repo_type="dataset", revision=revision)
    with open(path, newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != expected_columns:
            raise RuntimeError(f"{filename} has columns {reader.fieldnames}, expected {expected_columns}")
        return [{**row, "label": int(row["label"])} for row in reader]


def describe_split(name: str, rows: list[dict]) -> None:
    """Print columns, row count and label balance. Never prints a dialogue."""
    labels = Counter(row["label"] for row in rows)
    kinds = Counter((row["type"], row["label"]) for row in rows)
    print(f"{name}: {len(rows)} rows, columns {list(rows[0])}")
    print(f"  label balance: scam (1) = {labels[1]}, legitimate (0) = {labels[0]}")
    for (kind, label), count in sorted(kinds.items()):
        print(f"  type {kind:<14} label {label}: {count}")


def strip_tags(dialogue: str) -> str:
    """Remove the inline speaker tags and collapse whitespace. Live transcripts have none."""
    return " ".join(TAG_PATTERN.sub(" ", dialogue).split())


def strip_greeting(text: str) -> str:
    """Remove one leading greeting (hello, hi, hey, good morning / afternoon / evening) and the
    punctuation after it. The same rule for every row and both labels: in this dataset the
    greeting alone nearly gives the label away."""
    return GREETING_PATTERN.sub("", text, count=1)


def prepared(rows: list[dict]) -> list[dict]:
    """New rows: `text` has tags and the leading greeting removed; `original` keeps the greeting."""
    out = []
    for row in rows:
        original = strip_tags(row["dialogue"])
        out.append({"text": strip_greeting(original), "original": original, "type": row["type"], "label": row["label"]})
    return out


def greeting_counts(rows: list[dict]) -> dict:
    """How many prepared rows lost a greeting, in all and per label."""
    changed = Counter(row["label"] for row in rows if row["text"] != row["original"])
    return {"rows": len(rows), "greeting_removed": sum(changed.values()), "scam": changed[1], "legitimate": changed[0]}


def prefix_at(text: str, percent: int) -> str:
    """The first `percent` % of the text's words, rounded up, never fewer than one word."""
    words = text.split()
    keep = max(1, (len(words) * percent + 99) // 100)
    return " ".join(words[:keep])


def binary_metrics(labels: list[int], predictions: list[int]) -> dict:
    """Precision, recall and F1 for the scam class, with the counts behind them.

    A metric with a zero denominator is None, not 0, so no table can show a made-up number.
    """
    pairs = list(zip(labels, predictions))
    tp = sum(1 for label, guess in pairs if label == 1 and guess == 1)
    fp = sum(1 for label, guess in pairs if label == 0 and guess == 1)
    fn = sum(1 for label, guess in pairs if label == 1 and guess == 0)
    tn = sum(1 for label, guess in pairs if label == 0 and guess == 0)
    return {
        "n": len(pairs),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": tp / (tp + fp) if tp + fp else None,
        "recall": tp / (tp + fn) if tp + fn else None,
        "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,
    }


def correct_by_type(rows: list[dict], predictions: list[int]) -> dict:
    """Per scenario type: how many dialogues there are and how many were labelled correctly."""
    totals = Counter(row["type"] for row in rows)
    correct = Counter(row["type"] for row, guess in zip(rows, predictions) if guess == row["label"])
    return {kind: {"n": totals[kind], "correct": correct[kind]} for kind in sorted(totals)}


def rounded(value: float | None) -> float | None:
    """Four decimals for the results file; None stays None."""
    return None if value is None else round(value, 4)


def show(value: float | None) -> str:
    """A metric as text; an undefined one reads n/a."""
    return "n/a" if value is None else f"{value:.4f}"


def metric_rows(test: str, metrics: dict, context: str) -> list[dict]:
    """Table-ready rows for one test set: test, metric, value, n, notes (docs/SPEC.md section 8).

    n is each metric's own denominator: dialogues flagged as scam for precision, scam
    dialogues for recall, every dialogue scored for F1.
    """
    counts = f"tp={metrics['tp']} fp={metrics['fp']} fn={metrics['fn']} tn={metrics['tn']}"
    denominators = {
        "precision": (metrics["tp"] + metrics["fp"], "dialogues flagged as scam"),
        "recall": (metrics["tp"] + metrics["fn"], "scam dialogues"),
        "f1": (metrics["n"], "dialogues scored"),
    }
    rows = []
    for name in METRIC_NAMES:
        n, meaning = denominators[name]
        notes = f"n = {meaning}; {metrics['n']} scored in all; {counts}; {context}"
        rows.append({"test": test, "metric": name, "value": rounded(metrics[name]), "n": n, "notes": notes})
    return rows


def write_results(payload: dict, path: str) -> str:
    """Write one results file as indented JSON and return its path."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
    return path


def utc_now() -> str:
    """Current UTC time, to the second, for the results file."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
# 2. Load the test CSV from the Hugging Face hub and strip the speaker tags. Counts only.
test_raw = load_split(DATASET_REPO, DATASET_REVISION, TEST_FILE, EXPECTED_COLUMNS)
describe_split("test", test_raw)
if len(test_raw) != EXPECTED_TEST_ROWS:
    raise RuntimeError(f"test has {len(test_raw)} rows, expected {EXPECTED_TEST_ROWS}. Re-check docs/SPEC.md section 6.")

test_rows = prepared(test_raw)  # same preparation as the model: tags out, leading greeting out
if any(TAG_PATTERN.search(row["text"]) for row in test_rows):
    raise RuntimeError("Some dialogues still carry a speaker tag.")
GREETING_REMOVED = {"test_csv": greeting_counts(test_rows)}
print("speaker tags stripped from every test dialogue")
print(f"leading greeting removed: {GREETING_REMOVED}")

In [ ]:
# 3. The rule: any listed phrase present means SCAM.
# Case-insensitive, whole phrase, plural allowed, so "warrant" does not fire on "warranty".
def keyword_pattern(phrase: str) -> re.Pattern:
    """A whole-phrase matcher that tolerates extra spaces and a plural s."""
    words = [re.escape(word) for word in phrase.split()]
    return re.compile(r"\b" + r"\s+".join(words) + r"s?\b")


KEYWORD_PATTERNS = {phrase: keyword_pattern(phrase) for phrase in KEYWORDS}


def keyword_hits(text: str) -> list[str]:
    """Every listed phrase found in the text."""
    lowered = text.lower().replace("\u2019", "'")
    return [phrase for phrase, pattern in KEYWORD_PATTERNS.items() if pattern.search(lowered)]


def predict(text: str) -> int:
    """1 (scam) when any phrase is present, else 0."""
    return int(bool(keyword_hits(text)))


for text, expected in (("please buy two gift cards", 1), ("your warranty is expiring", 0), ("there is a WARRANT out", 1)):
    if predict(text) != expected:
        raise RuntimeError("The keyword rule failed its self-check.")
print("keyword rule self-check passed")

In [ ]:
# 4. Score the full dialogues and their first 40 % of words. Same tests and metrics as the model:
# the headline pair has the leading greeting removed, the second pair keeps the original text.
FULL_TEST = "bothbosu_test_full"
EARLY_TEST = f"bothbosu_test_prefix{EARLY_PERCENT}"
FULL_TEST_GREETING = f"{FULL_TEST}_with_greeting"
EARLY_TEST_GREETING = f"{EARLY_TEST}_with_greeting"
HEADLINE_TESTS = (FULL_TEST, EARLY_TEST)
test_labels = [row["label"] for row in test_rows]
test_sets = {
    FULL_TEST: [row["text"] for row in test_rows],
    EARLY_TEST: [prefix_at(row["text"], EARLY_PERCENT) for row in test_rows],
    FULL_TEST_GREETING: [row["original"] for row in test_rows],
    EARLY_TEST_GREETING: [prefix_at(row["original"], EARLY_PERCENT) for row in test_rows],
}

RESULTS = {}
for name, texts in test_sets.items():
    predictions = [predict(text) for text in texts]
    metrics = binary_metrics(test_labels, predictions)
    hits = {phrase: {"scam": 0, "legitimate": 0} for phrase in KEYWORDS}
    for text, label in zip(texts, test_labels):
        for phrase in keyword_hits(text):
            side = "scam" if label == 1 else "legitimate"
            hits[phrase] = {**hits[phrase], side: hits[phrase][side] + 1}
    RESULTS[name] = {"metrics": metrics, "by_type": correct_by_type(test_rows, predictions), "keyword_hits": hits}
    print(f"{name}: precision {show(metrics['precision'])}, recall {show(metrics['recall'])}, "
          f"F1 {show(metrics['f1'])}, n={metrics['n']} "
          f"(tp={metrics['tp']} fp={metrics['fp']} fn={metrics['fn']} tn={metrics['tn']})")
    for phrase, counts in hits.items():
        print(f"    {phrase:<20} dialogues hit: scam {counts['scam']}, legitimate {counts['legitimate']}")

In [ ]:
# 5. Write baselines.json, same shape as model.json.
context = "scam is the positive class; any listed phrase present"
early = f"first {EARLY_PERCENT} % of words"
removed = "leading greeting removed, as for the model"
kept = "original text with its greeting"
rows = metric_rows(FULL_TEST, RESULTS[FULL_TEST]["metrics"], f"{removed}; {context}")
rows = rows + metric_rows(EARLY_TEST, RESULTS[EARLY_TEST]["metrics"], f"{early}; {removed}; {context}")
rows = rows + metric_rows(FULL_TEST_GREETING, RESULTS[FULL_TEST_GREETING]["metrics"], f"{kept}; {context}")
rows = rows + metric_rows(EARLY_TEST_GREETING, RESULTS[EARLY_TEST_GREETING]["metrics"], f"{early}; {kept}; {context}")

payload = {
    "schema": RESULTS_SCHEMA,
    "system": "keyword",
    "created_utc": utc_now(),
    "dataset": {"repo": DATASET_REPO, "revision": DATASET_REVISION, "test_rows": len(test_raw)},
    "preprocessing": {
        "speaker_tags": "removed",
        "leading_greeting": "removed from the headline tests: hello, hi, hey, good morning / afternoon / "
                            "evening, plus the punctuation after it; one rule for every row and both labels",
        "greeting_removed": GREETING_REMOVED,
        "headline_tests": list(HEADLINE_TESTS),
    },
    "rule": "any listed phrase present; case-insensitive, whole phrase, plural allowed",
    "keywords": list(KEYWORDS),
    "rows": rows,
    "confusion": {name: {key: result["metrics"][key] for key in ("tp", "fp", "fn", "tn")} for name, result in RESULTS.items()},
    "by_type": {name: result["by_type"] for name, result in RESULTS.items()},
    "keyword_hits": {name: result["keyword_hits"] for name, result in RESULTS.items()},
    "caveats": ["Fixed phrase list from ml/README.md, set before the test split was scored. Not tuned."],
}

results_dir = os.path.join("eval", "results")
out_path = write_results(payload, os.path.join(results_dir if os.path.isdir(results_dir) else ".", "baselines.json"))
print(json.dumps(payload, indent=2))
print("\nwrote", os.path.abspath(out_path))

try:
    from google.colab import files

    files.download(out_path)
except ImportError:
    print("Not in Colab, so no browser download. The file above is the output.")